# Alquiler — 03 Feature engineering (Extraction → Transformation → Selection)

Transforma los artefactos de la limpieza (`rentals_train.csv` / `rentals_test.csv`,
split temporal 2021-2024 / 2025-2026) en la matriz lista para la regresión del
alquiler mensual. La pregunta que ordena el experimento es qué características de
la vivienda y de su ubicación explican `monthly_rent_rub`, y esa pregunta pone un
límite claro: ninguna variable construida desde la propia target —ni promedios de
alquileres por distrito o por mes, ni cocientes que la contengan— puede explicar
el alquiler, porque sería explicarlo con alquileres. La explicación tiene que
venir de la superficie, el edificio, el transporte, la identidad del lugar y el
contexto de otros mercados.

La target se modela **cruda, en rublos**, tal como la entrega la limpieza
(winsorizada en el entrenamiento, intacta en la evaluación). Estructura:

1. **Extraction** — candidatas derivadas fila a fila e inventario honesto de lo heredado.
2. **Transformation** — forma funcional de las continuas y codificación de las
   categóricas, elegidas con **walk-forward validation**: cada comparación se
   ajusta con el pasado y se evalúa en el año siguiente (2022, 2023 y 2024),
   en rublos, sin mezclar meses.
3. **Selection** — la matriz base que se le entrega al modelo, con su colinealidad medida.

Todo parámetro que se aprende (bases del one-hot, medias y desvíos del escalado)
se ajusta solo en train y se aplica a test.

## 0. Entorno

Correr con el kernel **(.venv)**. Si falta el entorno:

```
source .venv/bin/activate
pip install -r requirements.txt
```

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_DIR = Path("..") / "data"
OUT_DIR = DATA_DIR / "processed"
TARGET = "monthly_rent_rub"
Y = TARGET

RENOVATION_ORDER = ["no_renovation", "cosmetic", "euro", "designer"]
CATS_BASE = ["district", "building_type", "renovation", "metro_line"]

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("pandas", pd.__version__, "| numpy", np.__version__)
print("DATA_DIR:", DATA_DIR.resolve(), "| existe:", DATA_DIR.exists())

pandas 3.0.6 | numpy 2.5.4
DATA_DIR: /Users/tomasmontilla/workspace/facultad/Moscow-real-estate-market--Sales-and-rentals/data | existe: True


### Contrato de entrada

Se hereda de la limpieza sin rehacerla: división temporal ya hecha (el
entrenamiento termina el 31 de diciembre de 2024 y la evaluación empieza el 1 de
enero de 2025), target cruda en rublos —winsorizada solo en train—, contexto
mensual rezagado un mes y ningún `okrug`, que el distrito ya contiene. Las marcas
`target_winsorizado` y `flag_outlier_*` se quedan en los artefactos de la limpieza
como diagnóstico y no viajan a los de esta etapa: derivan del valor de la target
o señalan casos, no son características del aviso.

In [2]:
train = pd.read_csv(OUT_DIR / "rentals_train.csv", parse_dates=["date_posted"])
test = pd.read_csv(OUT_DIR / "rentals_test.csv", parse_dates=["date_posted"])

assert train.shape == (14820, 35), f"train inesperado: {train.shape}"
assert test.shape == (4841, 35), f"test inesperado: {test.shape}"
assert train["date_posted"].max() < pd.Timestamp("2025-01-01") <= test["date_posted"].min()
assert set(train["id"]).isdisjoint(test["id"]), "solape de id entre train y test"
assert "rent_per_sqm" not in train.columns and "log_rent" not in train.columns
print(f"train {len(train):,} x {train.shape[1]} | {train['date_posted'].min().date()} -> {train['date_posted'].max().date()}")
print(f"test  {len(test):,} x {test.shape[1]} | {test['date_posted'].min().date()} -> {test['date_posted'].max().date()}")
print(f"target cruda: skew train (winsorizada) = {float(stats.skew(train[TARGET])):.2f} | "
      f"skew test (intacta) = {float(stats.skew(test[TARGET])):.2f} | sin log ni copia transformada")
display(train[["id", "date_posted", "district", TARGET, "total_area", "metro_distance_min"]].head(3))

train 14,820 x 35 | 2021-01-01 -> 2024-12-31
test  4,841 x 35 | 2025-01-01 -> 2026-04-29
target cruda: skew train (winsorizada) = 0.80 | skew test (intacta) = 1.76 | sin log ni copia transformada


,id,date_posted,district,monthly_rent_rub,total_area,metro_distance_min
0,R000002,2022-11-12,Chertanovo Severnoye,37200.0,53.4,152
1,R000004,2024-10-11,Beskudnikovsky,30200.0,44.6,145
2,R000005,2023-07-11,Shchapovskoye,18700.0,56.7,472


## 1. Extraction — crear las candidatas derivadas

Extraer es construir columnas nuevas a partir de las existentes, todavía sin
decidir su forma definitiva ni si entran al modelo.

**La primera extracción es una resta.** Heredada de la limpieza viaja
`rental_price_per_sqm_monthly_lag1`, el precio medio de alquiler del distrito
cerrado el mes anterior. Es la variable que más correlaciona con la target
(r = 0,77) y justamente por eso sale también de estos artefactos: es un promedio
de alquileres, es decir, la target de otros avisos resumida. Un modelo que la usa
no aprende qué hace caro a un aviso; aprende que donde los alquileres son caros,
los alquileres son caros. El rezago la volvía legítima en el tiempo, pero no la
vuelve una característica de la vivienda ni de su ubicación.

Del resto de lo heredado se conserva como candidatas el contexto que no se
calcula desde alquileres: las variaciones mensuales de los precios de **venta**
del distrito, los volúmenes de avisos (conteos de publicaciones, no precios de
alquiler), la tasa clave, la distancia de la estación
al centro y su antigüedad. Y se crea una derivada fila a fila que la limpieza no
traía: la **edad del edificio** al momento de la publicación, diferencia entre el
año del aviso y el año de construcción, sin ningún parámetro aprendido.

In [3]:
# La derivada de la target sale de los artefactos de esta etapa (no solo de la matriz)
assert "rental_price_per_sqm_monthly_lag1" in train.columns
r_circular = float(train["rental_price_per_sqm_monthly_lag1"].corr(train[TARGET]))
train = train.drop(columns=["rental_price_per_sqm_monthly_lag1"])
test = test.drop(columns=["rental_price_per_sqm_monthly_lag1"])
assert "rental_price_per_sqm_monthly_lag1" not in train.columns
print(f"fuera rental_price_per_sqm_monthly_lag1 (r = {r_circular:.2f} con la target: promedio de alquileres, no caracteristica)")

for nombre, d in [("train", train), ("test", test)]:
    d["edad_edificio"] = d["date_posted"].dt.year - d["building_year"]
assert train["edad_edificio"].min() >= 0 and test["edad_edificio"].min() >= 0
print("creada en train y test (fila a fila): edad_edificio = anio de publicacion - building_year")

candidatas_contexto = [c for c in train.columns if c in [
    "secondary_mom_change_pct_lag1", "newbuild_mom_change_pct_lag1",
    "n_listings_secondary_lag1", "n_listings_newbuild_lag1", "n_listings_rental_lag1",
    "cbr_key_rate_pct_lag1", "station_to_center_km", "estacion_antiguedad",
    "floor", "total_floors", "edad_edificio", "lat", "lon",
    "rooms", "to_center_km", "metro_station"]]
print(f"candidatas que viajan fuera de la matriz base ({len(candidatas_contexto)}):", candidatas_contexto)
print("nunca predictoras: id, date_posted, year_month, target, target_winsorizado, flag_outlier_*")

fuera rental_price_per_sqm_monthly_lag1 (r = 0.77 con la target: promedio de alquileres, no caracteristica)
creada en train y test (fila a fila): edad_edificio = anio de publicacion - building_year
candidatas que viajan fuera de la matriz base (16): ['lat', 'lon', 'rooms', 'floor', 'total_floors', 'metro_station', 'to_center_km', 'secondary_mom_change_pct_lag1', 'newbuild_mom_change_pct_lag1', 'n_listings_secondary_lag1', 'n_listings_newbuild_lag1', 'n_listings_rental_lag1', 'cbr_key_rate_pct_lag1', 'station_to_center_km', 'estacion_antiguedad', 'edad_edificio']
nunca predictoras: id, date_posted, year_month, target, target_winsorizado, flag_outlier_*


## 2. Transformation — dar forma a cada candidata

### 2.1 Formas funcionales con walk-forward validation

Las dos continuas que llegan a la matriz son la superficie y los minutos al
metro. Ambas tienen cola a la derecha (skew 0,58 y 1,96 en crudo) y la pregunta
no es cómo se ven, sino qué forma predice mejor el año siguiente habiendo visto
solo los anteriores. Por eso la comparación es **walk-forward**: se ajusta con
2021 y se evalúa en 2022; se ajusta con 2021-2022 y se evalúa en 2023; se ajusta
con 2021-2023 y se evalúa en 2024. El error se mide en rublos sobre la target
cruda, que es la moneda en la que el modelo final va a acertar o errar.

> La regresión lineal de esta sección es solo una vara de medir entre
> transformaciones. El modelo del proyecto es el OLS de la etapa siguiente;
> acá no se valida ni se selecciona ese modelo.

El mismo procedimiento confirma, ya con la matriz casi armada, el destino de los
dos pares que comparten señal: la distancia al centro y los ambientes. Si una
columna no baja el error del año siguiente cuando su compañera ya está, no entra,
aunque su correlación simple prometa.

In [4]:
FORMAS = {"cruda": lambda s: s.astype(float), "log1p": np.log1p, "sqrt": np.sqrt}
ANIOS_VAL = [2022, 2023, 2024]

def construir_X(d, form_area, form_metro, enc, formas_centro=None, con_rooms=False,
                  cats=None, extra_num=("deposit_months", "furnished", "pets_allowed")):
    X = pd.DataFrame(index=d.index)
    X["area_f"] = FORMAS[form_area](d["total_area"])
    X["metro_f"] = FORMAS[form_metro](d["metro_distance_min"])
    if formas_centro:
        X["centro_f"] = FORMAS[formas_centro](d["to_center_km"])
    for c in extra_num:
        X[c] = d[c].astype(float)
    cats = (CATS_BASE if cats is None else list(cats)) + (["rooms"] if con_rooms and "rooms" not in (cats or []) else [])
    dum = pd.DataFrame(enc.transform(d[cats]), columns=enc.get_feature_names_out(cats), index=d.index)
    return pd.concat([X, dum], axis=1)

def walk_forward(form_area="log1p", form_metro="log1p", formas_centro=None, con_rooms=False,
                   cats=None, extra_num=("deposit_months", "furnished", "pets_allowed")):
    # Ajusta solo con el pasado y evalua en el anio siguiente. Devuelve RMSE/MAE/R2 por fold.
    cats = (CATS_BASE if cats is None else cats) + (["rooms"] if con_rooms else [])
    filas = []
    for anio in ANIOS_VAL:
        tr = train[train["date_posted"].dt.year < anio]
        va = train[train["date_posted"].dt.year == anio]
        bases = [tr[c].mode().iloc[0] for c in cats]
        enc = OneHotEncoder(handle_unknown="ignore", drop=bases, sparse_output=False).fit(tr[cats])
        Xtr, Xva = construir_X(tr, form_area, form_metro, enc, formas_centro, con_rooms, cats, extra_num), \
                   construir_X(va, form_area, form_metro, enc, formas_centro, con_rooms, cats, extra_num)
        cols_num = ["area_f", "metro_f"] + (["centro_f"] if formas_centro else [])
        sc = StandardScaler().fit(Xtr[cols_num])
        Xtr[cols_num], Xva[cols_num] = sc.transform(Xtr[cols_num]), sc.transform(Xva[cols_num])
        m = LinearRegression().fit(Xtr, tr[TARGET])
        pred = m.predict(Xva)
        filas.append({"validacion": anio, "n": len(va),
                      "RMSE_RUB": float(np.sqrt(mean_squared_error(va[TARGET], pred))),
                      "MAE_RUB": float(mean_absolute_error(va[TARGET], pred)),
                      "R2": float(r2_score(va[TARGET], pred))})
    return pd.DataFrame(filas)

configs = [(f"area {fa} | metro {fm}", fa, fm)
           for fa in ["cruda", "log1p", "sqrt"] for fm in ["cruda", "log1p", "sqrt"]]
resumen = []
for nombre, fa, fm in configs:
    wf = walk_forward(fa, fm)
    resumen.append({"configuracion": nombre, "RMSE_promedio": round(wf["RMSE_RUB"].mean(), 0),
                    "MAE_promedio": round(wf["MAE_RUB"].mean(), 0), "R2_promedio": round(wf["R2"].mean(), 4)})
    print(f"--- {nombre} ---"); display(wf.round({"RMSE_RUB": 0, "MAE_RUB": 0, "R2": 3}))
tabla_formas = pd.DataFrame(resumen).sort_values("RMSE_promedio")
display(tabla_formas)
MEJOR = tabla_formas.iloc[0]
FORM_AREA, FORM_METRO = [(fa, fm) for n, fa, fm in configs if n == MEJOR["configuracion"]][0]
print(f"forma elegida por walk-forward: area={FORM_AREA} | metro={FORM_METRO} "
      f"(menor RMSE promedio prediciendo el anio siguiente, en RUB)")

--- area cruda | metro cruda ---


,validacion,n,RMSE_RUB,MAE_RUB,R2
0,2022,3764,8388.0,6211.0,0.875
1,2023,3747,9789.0,7413.0,0.843
2,2024,3705,10022.0,7620.0,0.849


--- area cruda | metro log1p ---


,validacion,n,RMSE_RUB,MAE_RUB,R2
0,2022,3764,7602.0,5723.0,0.897
1,2023,3747,9098.0,7014.0,0.865
2,2024,3705,9430.0,7286.0,0.867


--- area cruda | metro sqrt ---


...

,validacion,n,RMSE_RUB,MAE_RUB,R2
0,2022,3764,7592.0,5704.0,0.897
1,2023,3747,9071.0,6961.0,0.865
2,2024,3705,9423.0,7261.0,0.867


--- area sqrt | metro sqrt ---


,validacion,n,RMSE_RUB,MAE_RUB,R2
0,2022,3764,8029.0,5997.0,0.885
1,2023,3747,9456.0,7214.0,0.854
2,2024,3705,9746.0,7484.0,0.857


,configuracion,RMSE_promedio,MAE_promedio,R2_promedio
7,area sqrt | metro log1p,8695.0,6642.0,0.8765
1,area cruda | metro log1p,8710.0,6674.0,0.8761
4,area log1p | metro log1p,8743.0,6647.0,0.8752
8,area sqrt | metro sqrt,9077.0,6898.0,0.8655
2,area cruda | metro sqrt,9091.0,6925.0,0.8650
5,area log1p | metro sqrt,9121.0,6907.0,0.8642
6,area sqrt | metro cruda,9384.0,7057.0,0.8562
0,area cruda | metro cruda,9399.0,7082.0,0.8558
3,area log1p | metro cruda,9426.0,7068.0,0.8550


forma elegida por walk-forward: area=sqrt | metro=log1p (menor RMSE promedio prediciendo el anio siguiente, en RUB)


In [5]:
# Confirmacion fuera de tiempo de los pares que comparten senal (sobre la forma elegida)
pares_wf = []
for nombre, fc, cr in [("base (area + metro)", None, False),
                       ("base + distancia al centro", "log1p", False),
                       ("base + ambientes (dummies)", None, True),
                       ("base + centro + ambientes", "log1p", True)]:
    wf = walk_forward(FORM_AREA, FORM_METRO, formas_centro=fc, con_rooms=cr)
    pares_wf.append({"matriz": nombre, "RMSE_promedio": round(wf["RMSE_RUB"].mean(), 0),
                     "MAE_promedio": round(wf["MAE_RUB"].mean(), 0), "R2_promedio": round(wf["R2"].mean(), 4)})
tabla_pares = pd.DataFrame(pares_wf)
display(tabla_pares)
_base = tabla_pares.iloc[0]; _amb = tabla_pares.iloc[2]
print("Regla de entrada: una columna entra solo si baja RMSE y MAE del anio siguiente con su companera adentro.")
print(f"centro: RMSE {_base['RMSE_promedio']:,.0f} -> {tabla_pares.iloc[1]['RMSE_promedio']:,.0f} (no baja) -> fuera, ya lo resume el distrito.")
print(f"ambientes: RMSE {_base['RMSE_promedio']:,.0f} -> {_amb['RMSE_promedio']:,.0f} y MAE {_base['MAE_promedio']:,.0f} -> {_amb['MAE_promedio']:,.0f} (no baja) -> fuera;")
print("con otra forma de superficie la diferencia era de un digito y mixta, asi que tampoco habia senal que pagar con 3 columnas.")

,matriz,RMSE_promedio,MAE_promedio,R2_promedio
0,base (area + metro),8695.0,6642.0,0.8765
1,base + distancia al centro,8697.0,6643.0,0.8765
2,base + ambientes (dummies),8697.0,6647.0,0.8765
3,base + centro + ambientes,8699.0,6648.0,0.8764


Regla de entrada: una columna entra solo si baja RMSE y MAE del anio siguiente con su companera adentro.
centro: RMSE 8,695 -> 8,697 (no baja) -> fuera, ya lo resume el distrito.
ambientes: RMSE 8,695 -> 8,697 y MAE 6,642 -> 6,647 (no baja) -> fuera;
con otra forma de superficie la diferencia era de un digito y mixta, asi que tampoco habia senal que pagar con 3 columnas.


### 2.2 Variables categóricas: one-hot k−1, con el distrito adentro

OLS no consume texto, y codificar etiquetas como 1, 2, 3 inventaría un orden y
unas distancias que no existen. El tratamiento es **one-hot con k−1 columnas**
para las cuatro categóricas de la matriz: `district`, `building_type`,
`renovation` y `metro_line`. La categoría que falta en cada variable es su
mayoritaria en train y queda como base: cada dummy mide la diferencia contra la
categoría más común, que es la comparación más estable.

El distrito es la decisión pesada de esta sección y se sostiene fuera de
tiempo, con la misma vara que eligió las formas: sin sus dummies el error del
año siguiente se dispara, y agregar la estación no lo baja de forma apreciable
(tabla siguiente). Por eso `metro_station` queda fuera como identidad: su
información ya vive en el distrito y en cuánto se tarda en llegar a ella.
Codificar el distrito por su alquiler promedio queda descartado por la misma
regla de la Extraction: sería colar la target promediada con nombre de código.

`renovation` tiene orden real, pero sus medias de alquiler no suben en pasos
iguales, así que un código 0-3 le impondría al modelo escalones parejos que los
datos no muestran. Se queda en dummies, que no asumen ningún espaciamiento.

In [6]:
geo_wf = []
for nombre, cats_g, fc in [("sin distrito (distancias + linea)", ["building_type", "renovation", "metro_line"], "log1p"),
                           ("con distrito", None, None),
                           ("con distrito + estacion", CATS_BASE + ["metro_station"], None)]:
    wf_g = walk_forward(FORM_AREA, FORM_METRO, formas_centro=fc, cats=cats_g,
                        extra_num=("furnished", "pets_allowed"))
    geo_wf.append({"bloque": nombre, "RMSE_promedio": round(wf_g["RMSE_RUB"].mean(), 0),
                   "MAE_promedio": round(wf_g["MAE_RUB"].mean(), 0), "R2_promedio": round(wf_g["R2"].mean(), 4)})
display(pd.DataFrame(geo_wf))
print("La comparacion es fuera de tiempo (walk-forward), la misma vara que eligio las formas:")
print("sin distrito el error se dispara; la estacion no lo baja de forma apreciable.")

t_ren = train.groupby("renovation", observed=True)[TARGET].agg(["mean", "count"]).reindex(RENOVATION_ORDER)
t_ren["salto_vs_anterior"] = t_ren["mean"].diff()
print("--- medias de alquiler por renovacion (orden real; saltos desiguales -> dummies, no codigo 0-3) ---")
display(t_ren.round(0))

CATS_BASE = ["district", "building_type", "renovation"]
bases = {c: train[c].mode().iloc[0] for c in CATS_BASE}
print("base (mayoritaria en train) por variable:", bases)
enc = OneHotEncoder(handle_unknown="ignore", drop=[bases[c] for c in CATS_BASE],
                    sparse_output=False).fit(train[CATS_BASE])
dum_train = pd.DataFrame(enc.transform(train[CATS_BASE]),
                         columns=enc.get_feature_names_out(CATS_BASE), index=train.index)
dum_test = pd.DataFrame(enc.transform(test[CATS_BASE]),
                        columns=enc.get_feature_names_out(CATS_BASE), index=test.index)
print(f"one-hot (k-1) ajustado en train: {dum_train.shape[1]} columnas desde {CATS_BASE}")
for c in CATS_BASE:
    cols_c = [n for n in dum_train.columns if n.startswith(c + "_")]
    s_tr = dum_train[cols_c].sum(axis=1)
    assert set(s_tr.unique()) <= {0.0, 1.0}, c
    nuevas = sorted(set(test[c].astype(str).unique()) - set(train[c].astype(str).unique()))
    print(f"{c}: {len(cols_c)} dummies | base '{bases[c]}' | categorias en test no vistas en train: "
          f"{nuevas if nuevas else 'ninguna'}")

,bloque,RMSE_promedio,MAE_promedio,R2_promedio
0,sin distrito (distancias + linea),12592.0,9362.0,0.7413
1,con distrito,8695.0,6642.0,0.8765
2,con distrito + estacion,8691.0,6622.0,0.8766


La comparacion es fuera de tiempo (walk-forward), la misma vara que eligio las formas:
sin distrito el error se dispara; la estacion no lo baja de forma apreciable.
--- medias de alquiler por renovacion (orden real; saltos desiguales -> dummies, no codigo 0-3) ---


,mean,count,salto_vs_anterior
renovation,,,
no_renovation,42480.0,1468,NaN
cosmetic,46738.0,6708,4258.0
euro,53061.0,5139,6323.0
designer,60755.0,1505,7693.0


base (mayoritaria en train) por variable: {'district': 'Khoroshyovo-Mnyovniki', 'building_type': 'panel', 'renovation': 'cosmetic'}
one-hot (k-1) ajustado en train: 136 columnas desde ['district', 'building_type', 'renovation']
district: 128 dummies | base 'Khoroshyovo-Mnyovniki' | categorias en test no vistas en train: ninguna
building_type: 5 dummies | base 'panel' | categorias en test no vistas en train: ninguna
renovation: 3 dummies | base 'cosmetic' | categorias en test no vistas en train: ninguna


In [7]:
# Aporte incremental fuera de tiempo, bloque por bloque (forma ya elegida)
bloques = []
pasos = [("distrito + superficie + metro (sin binarias)", ["district"], ()),
         ("+ amueblado y mascotas", ["district"], ("furnished", "pets_allowed")),
         ("+ tipo de edificio", ["district", "building_type"], ("furnished", "pets_allowed")),
         ("+ renovacion", ["district", "building_type", "renovation"], ("furnished", "pets_allowed")),
         ("+ linea de metro", ["district", "building_type", "renovation", "metro_line"], ("furnished", "pets_allowed")),
         ("+ meses de deposito", ["district", "building_type", "renovation", "metro_line"], ("furnished", "pets_allowed", "deposit_months"))]
for nombre, cats_b, extra_b in pasos:
    wf_b = walk_forward(FORM_AREA, FORM_METRO, cats=cats_b, extra_num=extra_b)
    bloques.append({"bloque": nombre, "RMSE_promedio": round(wf_b["RMSE_RUB"].mean(), 0),
                    "MAE_promedio": round(wf_b["MAE_RUB"].mean(), 0)})
tabla_bloques = pd.DataFrame(bloques)
tabla_bloques["delta_RMSE"] = tabla_bloques["RMSE_promedio"].diff()
display(tabla_bloques)
print("tipo de edificio y renovacion bajan el error de forma clara; la linea de metro y el deposito")
print("no lo bajan de forma apreciable (diferencias de un digito, inconsistentes entre anios): quedan")
print("fuera de la matriz por parsimonia. Amueblado y mascotas bajan el error en la segunda fila: adentro.")

,bloque,RMSE_promedio,MAE_promedio,delta_RMSE
0,distrito + superficie + metro (sin binarias),11396.0,8606.0,NaN
1,+ amueblado y mascotas,11255.0,8494.0,-141.0
2,+ tipo de edificio,10248.0,7698.0,-1007.0
3,+ renovacion,8699.0,6651.0,-1549.0
4,+ linea de metro,8695.0,6642.0,-4.0
5,+ meses de deposito,8695.0,6642.0,0.0


tipo de edificio y renovacion bajan el error de forma clara; la linea de metro y el deposito
no lo bajan de forma apreciable (diferencias de un digito, inconsistentes entre anios): quedan
fuera de la matriz por parsimonia. Amueblado y mascotas bajan el error en la segunda fila: adentro.


### 2.3 Escalado: mismos coeficientes comparables, mismo ajuste

Estandarizar las dos continuas de la matriz (restar la media y dividir por el
desvío **de train**) no cambia ni una predicción del OLS: reescala los
coeficientes. Su sentido es comparativo —un desvío más de superficie contra un
desvío más de minutos al metro, en la misma unidad— y práctico para cualquier
procedimiento posterior que compare o penalice coeficientes. Las dummies y las
binarias ya viven en {0, 1} y no se tocan; `deposit_months` (0 a 2) queda en su
escala corta. La target no se escala ni se transforma: se predice en rublos.

In [8]:
train["area_f"] = FORMAS[FORM_AREA](train["total_area"])
test["area_f"] = FORMAS[FORM_AREA](test["total_area"])
train["metro_f"] = FORMAS[FORM_METRO](train["metro_distance_min"])
test["metro_f"] = FORMAS[FORM_METRO](test["metro_distance_min"])
print(f"skew con la forma elegida (train): area {float(stats.skew(train['area_f'])):.2f} | "
      f"metro {float(stats.skew(train['metro_f'])):.2f}")

CONT_Z = ["area_f", "metro_f"]
scaler = StandardScaler().fit(train[CONT_Z])
z_train = pd.DataFrame(scaler.transform(train[CONT_Z]),
                       columns=[c + "_z" for c in CONT_Z], index=train.index)
z_test = pd.DataFrame(scaler.transform(test[CONT_Z]),
                      columns=[c + "_z" for c in CONT_Z], index=test.index)
params = pd.DataFrame({"media_train (usada)": scaler.mean_,
                       "desvio_train (usado)": scaler.scale_,
                       "media_global (prohibida)": pd.concat([train, test])[CONT_Z].mean().to_numpy()},
                      index=CONT_Z)
display(params.round(3))
assert np.allclose(scaler.mean_, train[CONT_Z].mean().to_numpy())
print("OK: el escalado aprendio de train (14.820); la media global solo se exhibe, no se usa.")
print("media de cada z en train (~0):", np.round(z_train.mean().to_numpy(), 6),
      "| desvio (~1):", np.round(z_train.std(ddof=0).to_numpy(), 6))

skew con la forma elegida (train): area 0.31 | metro -0.56


,media_train (usada),desvio_train (usado),media_global (prohibida)
area_f,7.393,0.843,7.393
metro_f,4.397,1.279,4.390


OK: el escalado aprendio de train (14.820); la media global solo se exhibe, no se usa.
media de cada z en train (~0): [-0.  0.] | desvio (~1): [1. 1.]


## 3. Selection — qué entra a la matriz y qué queda fuera

La matriz base reúne lo que sobrevivió a las mediciones anteriores y nada más:

- **Geografía por identidad:** 128 dummies de `district`, que cargan el nivel del
  barrio sin promediar alquileres.
- **Vivienda y edificio:** superficie y minutos al metro en la forma elegida por
  walk-forward, estandarizadas; `furnished` y `pets_allowed` en su escala;
  5 dummies de `building_type` y 3 de `renovation`.
- **Sin aporte medido, afuera:** las 11 dummies de `metro_line` y `deposit_months`
  no bajaron el error del año siguiente; sumarlas sería pagar columnas por ruido.

Quedan fuera de la matriz, viajando en los artefactos como candidatas para que la
etapa de modelo decida si abre el pool: pisos del aviso y del edificio, año de
construcción y edad del edificio, distancia y antigüedad de la estación,
variaciones de precios de venta y volúmenes de avisos rezagados, tasa clave,
coordenadas, ambientes, distancia al centro y la identidad de la estación.
Afuera de los artefactos, directamente, quedó el precio medio de alquiler del
distrito: la única columna que explicaba el alquiler con alquileres.

Antes de guardar se mide la colinealidad de la matriz completa. Con 140 columnas
el chequeo que importa es que ninguna sea casi combinación de las demás, porque
eso infla la varianza de su coeficiente hasta volverlo ilegible.

In [9]:
ZCOLS = ["area_f_z", "metro_f_z"]
DUMMIES = list(dum_train.columns)
FEATURES_BASE = ZCOLS + ["furnished", "pets_allowed"] + DUMMIES

fin_train = pd.concat([z_train, train[["furnished", "pets_allowed"]], dum_train], axis=1)[FEATURES_BASE]
fin_test = pd.concat([z_test, test[["furnished", "pets_allowed"]], dum_test], axis=1)[FEATURES_BASE]
assert list(fin_train.columns) == FEATURES_BASE and list(fin_test.columns) == FEATURES_BASE
assert fin_train.isna().sum().sum() == 0 and fin_test.isna().sum().sum() == 0
PROHIBIDAS = ["rental_price_per_sqm_monthly_lag1", "rent_per_sqm", TARGET, "target_winsorizado",
              "flag_outlier_total_area", "flag_outlier_metro_distance_min",
              "flag_outlier_to_center_km", "id", "date_posted"]
assert not [c for c in FEATURES_BASE if c in PROHIBIDAS]
print(f"matriz base: train {fin_train.shape} | test {fin_test.shape} | p = {len(FEATURES_BASE)} predictores")
print(f"composicion: {len(DUMMIES)} dummies (128 distrito + 8 edificio/renovacion) "
      f"+ 2 continuas estandarizadas + 2 binarias")

# VIF desde la inversa de la correlacion (equivalente, rapido para p grande)
corr_inv = np.linalg.pinv(np.corrcoef(fin_train.to_numpy(), rowvar=False))
vifs = pd.Series(np.diag(corr_inv), index=FEATURES_BASE).sort_values(ascending=False)
print("VIF mas altos de la matriz base (train):")
display(vifs.head(8).round(2).rename("VIF").to_frame())
print(f"VIF maximo: {vifs.max():.2f} | predictores con VIF > 10: {int((vifs > 10).sum())}")

sane = sm.OLS(train[TARGET], sm.add_constant(fin_train, has_constant="add")).fit()
wf_base = walk_forward(FORM_AREA, FORM_METRO)
print(f"sanidad (no es el modelo final): adj_R2 en train = {sane.rsquared_adj:.4f} | "
      f"walk-forward R2 = {wf_base['R2'].mean():.4f}, RMSE = {wf_base['RMSE_RUB'].mean():,.0f} RUB")
print("Ojo: ese walk-forward reutiliza los folds que eligieron la forma, asi que es una cota optimista;")
print("y se midio sobre la target winsorizada de train. En test la target es cruda y el error sera mayor.")

matriz base: train (14820, 140) | test (4841, 140) | p = 140 predictores
composicion: 136 dummies (128 distrito + 8 edificio/renovacion) + 2 continuas estandarizadas + 2 binarias
VIF mas altos de la matriz base (train):


,VIF
metro_f_z,2.87
district_Dorogomilovo,1.98
district_Klyonovskoye,1.98
district_Brateyevo,1.97
district_Krasnopakhorskoye,1.97
district_Novofyodorovskoye,1.97
district_Kotlovka,1.97
district_Zyablikovo,1.96


VIF maximo: 2.87 | predictores con VIF > 10: 0
sanidad (no es el modelo final): adj_R2 en train = 0.8988 | walk-forward R2 = 0.8764, RMSE = 8,699 RUB
Ojo: ese walk-forward reutiliza los folds que eligieron la forma, asi que es una cota optimista;
y se midio sobre la target winsorizada de train. En test la target es cruda y el error sera mayor.


### Guardado

Los artefactos de esta etapa conservan las columnas originales que superaron la
Extraction —incluidas las candidatas que no entraron a la matriz— junto con las
formas transformadas, las estandarizadas y las dummies. La lista
`FEATURES_BASE` es el contrato: eso, y solo eso, es la matriz. Los artefactos de
la limpieza no se tocaron.

In [10]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
NO_VIAJAN = ["target_winsorizado", "flag_outlier_total_area", "flag_outlier_metro_distance_min",
             "flag_outlier_to_center_km", "building_year"]
train_fe = pd.concat([train.drop(columns=[c for c in ZCOLS + DUMMIES + ["area_f", "metro_f"] + NO_VIAJAN if c in train.columns]),
                      train[["area_f", "metro_f"]], z_train, dum_train], axis=1)
test_fe = pd.concat([test.drop(columns=[c for c in ZCOLS + DUMMIES + ["area_f", "metro_f"] + NO_VIAJAN if c in test.columns]),
                     test[["area_f", "metro_f"]], z_test, dum_test], axis=1)
train_fe.to_csv(OUT_DIR / "rentals_train_fe.csv", index=False)
test_fe.to_csv(OUT_DIR / "rentals_test_fe.csv", index=False)
for f in ["rentals_train_fe.csv", "rentals_test_fe.csv"]:
    d = pd.read_csv(OUT_DIR / f)
    for prohibida in ["rental_price_per_sqm_monthly_lag1", "rent_per_sqm"] + NO_VIAJAN:
        assert prohibida not in d.columns, prohibida
    presentes = [c for c in FEATURES_BASE if c in d.columns]
    print(f"{f}: {d.shape[0]:,} x {d.shape[1]} | features de la matriz presentes: {len(presentes)}/{len(FEATURES_BASE)} "
          f"| sin derivadas de la target ni marcas de diagnostico")
print("FEATURES_BASE (conteo):", len(FEATURES_BASE), "| primeras:", FEATURES_BASE[:8])

rentals_train_fe.csv: 14,820 x 170 | features de la matriz presentes: 140/140 | sin derivadas de la target ni marcas de diagnostico
rentals_test_fe.csv: 4,841 x 170 | features de la matriz presentes: 140/140 | sin derivadas de la target ni marcas de diagnostico
FEATURES_BASE (conteo): 140 | primeras: ['area_f_z', 'metro_f_z', 'furnished', 'pets_allowed', 'district_Aeroport', 'district_Akademichesky', 'district_Alekseevsky', 'district_Altufyevsky']


## Resumen — Extraction → Transformation → Selection

| Variable | Resolución | Motivo medido |
|---|---|---|
| `monthly_rent_rub` | target cruda en RUB | la forma del precio; errores en la moneda del problema |
| `rental_price_per_sqm_monthly_lag1` | **fuera de los artefactos** | promedio de alquileres: explica el alquiler con alquileres (r = 0,77) |
| `district` | one-hot k−1 (128, base mayoritaria) | sin distrito el error walk-forward se dispara |
| `metro_station` | fuera como identidad | +0,000 sobre `district` |
| `total_area` | forma elegida por walk-forward, estandarizada | menor RMSE prediciendo el año siguiente |
| `metro_distance_min` | forma elegida por walk-forward, estandarizada | idem; le gana a la distancia al centro con el distrito adentro |
| `to_center_km` | fuera de la matriz (candidata) | el distrito absorbe la centralidad |
| `rooms` | fuera de la matriz (candidata) | evidencia mixta (−21 RUB de RMSE, +1 de MAE): fuera por parsimonia |
| `building_type`, `renovation` | one-hot k−1 (base mayoritaria) | bajan el error fuera de tiempo; en `renovation` los saltos no son iguales |
| `metro_line`, `deposit_months` | fuera de la matriz (candidatas) | no bajan el error del año siguiente: columnas por ruido |
| resto heredado | candidatas en los artefactos | señales débiles; ojo: `lat`/`lon` repiten al distrito si el pool se abre sin mirar colinealidad |
| `id`, fechas, marcas | nunca predictoras | trazabilidad y diagnóstico, no características |

**Qué se le entrega a la siguiente etapa:** `rentals_train_fe.csv` y
`rentals_test_fe.csv` con la matriz `FEATURES_BASE` (p = 140) embebida y las
candidatas a la vista. La selección final entre esas 152 columnas, la eventual
apertura del pool de candidatas y el diagnóstico de residuos son trabajo del
modelo; esta etapa solo garantiza que la matriz no contiene nada derivado de la
target, que ningún parámetro se aprendió del futuro y que cada bloque que entró
bajó el error prediciendo un año que no había visto. La etapa de modelo vigente
todavía espera el esquema anterior (target en log y otra matriz): deberá
actualizarse antes de consumir estos artefactos, y allí quedará por resolver
la tendencia temporal, que ninguna columna de esta base captura.